# 面试问题：怎样从零实现 CLIP 双编码器与对称图文对比损失？

## 可直接复述的回答主线

1. CLIP 用图像塔和文本塔分别编码，再投影到同一维度并做 L2 归一化。
2. 图文相似度是带可学习温度的 cosine 矩阵，batch 对角线代表匹配 pair。
3. 训练损失同时计算 image-to-text 和 text-to-image 交叉熵，避免只优化单向检索。
4. 图像塔可用手写 CNN，文本塔可用 token embedding 与手写 self-attention，不能直接导入预训练 CLIP。
5. 评测应展示原始图像、caption token、相似度矩阵、双向逐查询排名、Recall@1 和温度。
6. 不做 L2 normalize 时，大范数候选会靠长度而不是方向赢得点积，检索分数不可比。
7. 生产还需多正例处理、海量负样本、版权与安全过滤、向量索引、校准和模态漂移监控。

后续实验会在同一批可读输入上依次展示基线、手写核心机制、训练过程、逐样本结果、失败修正与生产边界。

## 1. 真实案例与输入预览

案例是 6 个仓储标签的灰度图与中文说明配对：横向条码、纵向条码、主对角标、副对角标、十字定位标和边框标签。所有图像被校正到相同全局均值，caption 长度也相同，因此只看亮度/长度的基线无法完成图文检索。

In [1]:
import math  # 计算 attention 缩放和训练梯度。
import warnings  # 过滤本地 PyTorch 环境的无关兼容警告。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦跨模态实验。
import torch  # 使用基础卷积、线性层和张量运算手写 CLIP。
torch.manual_seed(421)  # 固定双编码器初始化与训练轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程提高可复现性。
captions = ["横向 条码", "纵向 条码", "主对角 标记", "副对角 标记", "十字 定位", "边框 标签"]  # 定义六条可读仓储标签说明。
label_names = ["横向条码", "纵向条码", "主对角标", "副对角标", "十字定位标", "边框标签"]  # 定义每对图文的语义 ID。
base_patterns = []  # 保存六张二值空间图案。
horizontal = torch.zeros(16, 16)  # 初始化横向条码图。
horizontal[7:9, :] = 1.0  # 写入两像素高横条。
base_patterns.append(horizontal)  # 保存横向图案。
vertical = torch.zeros(16, 16)  # 初始化纵向条码图。
vertical[:, 7:9] = 1.0  # 写入两像素宽竖条。
base_patterns.append(vertical)  # 保存纵向图案。
main_diagonal = torch.zeros(16, 16)  # 初始化主对角标。
for index in range(16):  # 沿主对角线逐行绘制。
    main_diagonal[index, max(0, index - 1):min(16, index + 1)] = 1.0  # 写入近似两像素宽斜线。
base_patterns.append(main_diagonal)  # 保存主对角图案。
anti_diagonal = torch.zeros(16, 16)  # 初始化副对角标。
for index in range(16):  # 沿副对角线逐行绘制。
    column = 15 - index  # 计算当前行副对角列。
    anti_diagonal[index, max(0, column - 1):min(16, column + 1)] = 1.0  # 写入近似两像素宽斜线。
base_patterns.append(anti_diagonal)  # 保存副对角图案。
cross = torch.zeros(16, 16)  # 初始化十字定位标。
cross[7:9, 3:13] = 1.0  # 写入十字横臂。
cross[3:13, 7:9] = 1.0  # 写入十字竖臂。
base_patterns.append(cross)  # 保存十字图案。
border = torch.zeros(16, 16)  # 初始化边框标签。
border[3:13, 3] = 1.0  # 写入左边框。
border[3:13, 12] = 1.0  # 写入右边框。
border[3, 3:13] = 1.0  # 写入上边框。
border[12, 3:13] = 1.0  # 写入下边框。
base_patterns.append(border)  # 保存边框图案。
images = torch.stack([(pattern - pattern.mean() + 0.20).unsqueeze(0) for pattern in base_patterns])  # 校正六图全局均值并堆叠为 NCHW。
vocabulary = {"<PAD>": 0, "<BOS>": 1, "<EOS>": 2}  # 初始化文本特殊词表。
for caption in captions:  # 扫描六条中文 caption 构建词表。
    for token in caption.split():  # 逐空格 token 分配编号。
        if token not in vocabulary:  # 仅添加首次出现 token。
            vocabulary[token] = len(vocabulary)  # 使用当前词表长度作为编号。
text_ids = torch.tensor([[vocabulary["<BOS>"], vocabulary[first], vocabulary[second], vocabulary["<EOS>"]] for first, second in [caption.split() for caption in captions]], dtype=torch.long)  # 构造六乘四定长 caption token。
text_mask = torch.ones_like(text_ids, dtype=torch.bool)  # 标记四个 token 均有效。
def render_image(image):  # 把仓储标签图转为字符画。
    return "\n".join("".join("#" if float(pixel) > 0.5 else "." for pixel in row) for row in image.squeeze(0))  # 用阈值展示空间形状。
inverse_vocabulary = {index: token for token, index in vocabulary.items()}  # 建立编号到可读 token 的映射。
print("教学实验输入：6对仓储标签图文，image/text shapes=", tuple(images.shape), tuple(text_ids.shape))  # 展示图像和文本批次形状。
for index in range(len(images)):  # 逐 pair 展示图、caption 和 token 编号。
    readable_tokens = [inverse_vocabulary[token] for token in text_ids[index].tolist()]  # 还原当前 caption token。
    print(f"\npair-{index} {label_names[index]} caption={captions[index]} tokens={readable_tokens} mean={images[index].mean().item():.4f}\n{render_image(images[index])}")  # 输出真实图文输入。

教学实验输入：6对仓储标签图文，image/text shapes= (6, 1, 16, 16) (6, 4)

pair-0 横向条码 caption=横向 条码 tokens=['<BOS>', '横向', '条码', '<EOS>'] mean=0.2000
................
................
................
................
................
................
................
################
################
................
................
................
................
................
................
................

pair-1 纵向条码 caption=纵向 条码 tokens=['<BOS>', '纵向', '条码', '<EOS>'] mean=0.2000
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......
.......##.......

pair-2 主对角标 caption=主对角 标记 tokens=['<BOS>', '主对角', '标记', '<EOS>'] mean=0.2000
#...............
##..............
.##.............
..##............
...##...........
....##..........
.....##.........
......##........
.......##.......
........##...

## 2. Baseline / 基线：全局亮度对 caption 长度

图像全部均值 0.20，文本全部为 BOS+两个词+EOS，统计特征完全相同。用负绝对差构造相似度矩阵时六列并列，每个查询都会返回第一条 caption/图像。

In [2]:
image_brightness = images.mean(dim=(1, 2, 3))  # 提取每张图唯一的全局亮度。
caption_statistics = torch.full((len(captions),), 0.20)  # 把相同 caption 长度映射为相同统计量。
baseline_similarity = -torch.abs(image_brightness[:, None] - caption_statistics[None, :])  # 计算亮度与长度统计相似度。
baseline_image_to_text = baseline_similarity.argmax(dim=1)  # 为每张图选择最高分 caption。
baseline_text_to_image = baseline_similarity.argmax(dim=0)  # 为每条 caption 选择最高分图像。
pair_indices = torch.arange(len(captions))  # 构造正确 pair 的对角索引。
baseline_i2t_recall = float((baseline_image_to_text == pair_indices).to(torch.float32).mean().item())  # 计算 image-to-text Recall@1。
baseline_t2i_recall = float((baseline_text_to_image == pair_indices).to(torch.float32).mean().item())  # 计算 text-to-image Recall@1。
print("Baseline similarity matrix=", baseline_similarity.tolist())  # 展示统计特征造成的全并列矩阵。
for index in range(len(images)):  # 逐图展示错误检索结果。
    print(f"image={label_names[index]:<8} retrieved_text={label_names[baseline_image_to_text[index]]:<8} correct={bool(baseline_image_to_text[index] == index)}")  # 输出图找文 top1。
print(f"Baseline I2T R@1={baseline_i2t_recall:.4f}，T2I R@1={baseline_t2i_recall:.4f}")  # 展示双向同指标基线。

Baseline similarity matrix= [[-2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08], [-1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08], [-4.470348358154297e-08, -4.470348358154297e-08, -4.470348358154297e-08, -4.470348358154297e-08, -4.470348358154297e-08, -4.470348358154297e-08], [-2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08], [-1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08, -1.4901161193847656e-08], [-2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08, -2.9802322387695312e-08]]
image=横向条码     retrieved_text=横向条码     correct=True
image=纵向条码   

## 3. 底层实现：CNN 图像塔、手写文本注意力、投影归一化与对称 InfoNCE

不导入 OpenCLIP、Transformers 或现成视觉架构。文本 attention 明确计算 QKᵀ/√d、key mask、softmax 和 AV；两个塔分别投影后 L2 normalize，再乘可学习温度倒数。

In [3]:
class ImageEncoder(torch.nn.Module):  # 定义三层卷积仓储标签编码器。
    def __init__(self):  # 初始化局部特征网络。
        super().__init__()  # 注册卷积参数。
        self.features = torch.nn.Sequential(torch.nn.Conv2d(1, 8, 3, padding=1), torch.nn.SiLU(), torch.nn.MaxPool2d(2), torch.nn.Conv2d(8, 16, 3, padding=1), torch.nn.SiLU(), torch.nn.MaxPool2d(2), torch.nn.Conv2d(16, 24, 3, padding=1), torch.nn.SiLU())  # 把十六像素图变为四乘四二十四通道特征。
    def forward(self, inputs, return_debug=False):  # 提取空间形状并执行全局池化。
        feature_map = self.features(inputs)  # 生成四乘四局部视觉表示。
        pooled = feature_map.mean(dim=(2, 3))  # 对空间维求均值得到图像向量。
        return (pooled, feature_map) if return_debug else pooled  # 按需返回卷积特征图。
class TextSelfAttention(torch.nn.Module):  # 定义不调用现成 attention 的多头文本层。
    def __init__(self, dimension=24, heads=4):  # 初始化 QKV 与输出投影。
        super().__init__()  # 注册注意力参数。
        self.dimension = dimension  # 保存文本隐藏维度。
        self.heads = heads  # 保存注意力头数。
        self.head_dim = dimension // heads  # 计算每头子空间维度。
        self.qkv = torch.nn.Linear(dimension, 3 * dimension)  # 一次生成 query、key 和 value。
        self.output = torch.nn.Linear(dimension, dimension)  # 合并多头上下文。
    def forward(self, hidden, valid_mask):  # 执行双向 self-attention。
        batch_size, token_count, _ = hidden.shape  # 读取批量与文本长度。
        qkv = self.qkv(hidden).view(batch_size, token_count, 3, self.heads, self.head_dim)  # 切分 QKV 与 head。
        qkv = qkv.permute(2, 0, 3, 1, 4)  # 调整为三乘批次乘头乘 token 乘维度。
        queries, keys, values = qkv.unbind(dim=0)  # 解包三类注意力向量。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放 token 相似度。
        visible_keys = valid_mask[:, None, None, :]  # 构造 padding key 可见性 mask。
        attention = torch.softmax(scores.masked_fill(~visible_keys, -1.0e4), dim=-1)  # 屏蔽 padding 后归一化。
        context = attention @ values  # 按权重聚合 value。
        context = context.transpose(1, 2).contiguous().view(batch_size, token_count, self.dimension)  # 合并多头上下文。
        return self.output(context), attention  # 返回 token 更新与完整注意力矩阵。
class TextEncoder(torch.nn.Module):  # 定义 embedding、位置、attention 和 EOS pooling 文本塔。
    def __init__(self, vocab_size, dimension=24):  # 初始化文本编码参数。
        super().__init__()  # 注册文本塔参数。
        self.embedding = torch.nn.Embedding(vocab_size, dimension, padding_idx=0)  # 把 token 编号映射为向量。
        self.position = torch.nn.Parameter(torch.zeros(1, 4, dimension))  # 为四个文本位置添加可学习顺序。
        self.norm_one = torch.nn.LayerNorm(dimension)  # 为 attention 执行 pre-norm。
        self.attention = TextSelfAttention(dimension, heads=4)  # 创建手写多头文本注意力。
        self.norm_two = torch.nn.LayerNorm(dimension)  # 为逐 token MLP 执行 pre-norm。
        self.mlp = torch.nn.Sequential(torch.nn.Linear(dimension, 48), torch.nn.GELU(), torch.nn.Linear(48, dimension))  # 创建逐 token 特征变换。
    def forward(self, token_ids, valid_mask, return_debug=False):  # 编码 caption 并读取 EOS 表示。
        hidden = self.embedding(token_ids) + self.position[:, :token_ids.shape[1]]  # 合并 token 内容和位置。
        attention_update, attention = self.attention(self.norm_one(hidden), valid_mask)  # 计算双向文本上下文。
        hidden = hidden + attention_update  # 添加 attention 残差。
        hidden = hidden + self.mlp(self.norm_two(hidden))  # 添加逐 token MLP 残差。
        sentence = hidden[:, 3]  # 读取固定第四位置 EOS 的句向量。
        return (sentence, {"hidden": hidden, "attention": attention}) if return_debug else sentence  # 按需返回文本中间量。
class CLIPDualEncoder(torch.nn.Module):  # 定义图像塔、文本塔、共享空间和温度。
    def __init__(self, vocab_size, shared_dim=20):  # 初始化双塔与投影头。
        super().__init__()  # 注册跨模态模型参数。
        self.image_encoder = ImageEncoder()  # 创建手写 CNN 图像塔。
        self.text_encoder = TextEncoder(vocab_size)  # 创建手写 attention 文本塔。
        self.image_projection = torch.nn.Linear(24, shared_dim, bias=False)  # 把视觉特征投影到共享空间。
        self.text_projection = torch.nn.Linear(24, shared_dim, bias=False)  # 把文本特征投影到共享空间。
        self.logit_scale = torch.nn.Parameter(torch.tensor(math.log(1.0 / 0.07)))  # 以 log 形式保存可学习温度倒数。
    def encode_image(self, inputs):  # 生成单位长度图像 embedding。
        return torch.nn.functional.normalize(self.image_projection(self.image_encoder(inputs)), dim=1, eps=1.0e-8)  # 投影后执行 L2 归一化。
    def encode_text(self, token_ids, valid_mask):  # 生成单位长度文本 embedding。
        return torch.nn.functional.normalize(self.text_projection(self.text_encoder(token_ids, valid_mask)), dim=1, eps=1.0e-8)  # 投影后执行 L2 归一化。
    def forward(self, image_inputs, token_ids, valid_mask, return_debug=False):  # 计算完整图文相似度矩阵。
        image_features, image_map = self.image_encoder(image_inputs, return_debug=True)  # 取得视觉 pooled 与空间特征。
        text_features, text_debug = self.text_encoder(token_ids, valid_mask, return_debug=True)  # 取得 EOS 向量与注意力。
        image_embeddings = torch.nn.functional.normalize(self.image_projection(image_features), dim=1, eps=1.0e-8)  # 归一化图像共享向量。
        text_embeddings = torch.nn.functional.normalize(self.text_projection(text_features), dim=1, eps=1.0e-8)  # 归一化文本共享向量。
        scale = self.logit_scale.clamp(math.log(0.01), math.log(100.0)).exp()  # 限制温度倒数防止指数溢出。
        logits = scale * image_embeddings @ text_embeddings.T  # 计算图找文带温度 cosine 矩阵。
        debug = {"image_map": image_map, "text_hidden": text_debug["hidden"], "text_attention": text_debug["attention"], "image_embeddings": image_embeddings, "text_embeddings": text_embeddings, "scale": scale}  # 汇总双塔中间量。
        return (logits, debug) if return_debug else logits  # 按需返回图文 embedding 与注意力。
def symmetric_clip_loss(logits):  # 计算 image-to-text 与 text-to-image 对称 InfoNCE。
    targets = torch.arange(logits.shape[0])  # 构造正确 pair 的对角标签。
    image_to_text = torch.nn.functional.cross_entropy(logits, targets)  # 让每张图检索对应 caption。
    text_to_image = torch.nn.functional.cross_entropy(logits.T, targets)  # 让每条 caption 检索对应图像。
    return 0.5 * (image_to_text + text_to_image), image_to_text, text_to_image  # 返回总损失与两个方向分项。
model = CLIPDualEncoder(len(vocabulary))  # 创建待训练双编码器。
optimizer = torch.optim.Adam(model.parameters(), lr=0.012)  # 创建图像、文本、投影和温度优化器。
history = []  # 保存真实 backward 的对比训练轨迹。
for step in range(360):  # 在六个一一对应 pair 上执行全批次训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步全部参数梯度。
    logits, training_debug = model(images, text_ids, text_mask, return_debug=True)  # 前向编码两种模态并计算相似度矩阵。
    loss, i2t_loss, t2i_loss = symmetric_clip_loss(logits)  # 计算双向对称 InfoNCE。
    loss.backward()  # 对两个编码塔、投影头和温度真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度二范数。
    optimizer.step()  # 应用 Adam 更新 CLIP 参数。
    if step % 90 == 0 or step == 359:  # 每九十步保存训练证据。
        diagonal_recall = float((logits.argmax(dim=1) == torch.arange(len(images))).to(torch.float32).mean().item())  # 计算当前图找文 R@1。
        history.append({"step": step, "loss": loss.item(), "i2t_loss": i2t_loss.item(), "t2i_loss": t2i_loss.item(), "i2t_r1": diagonal_recall, "scale": training_debug["scale"].item(), "gradient_norm": gradient_norm})  # 保存损失、检索、温度和梯度。
model.eval()  # 切换到确定性图文推理模式。
with torch.no_grad():  # 取得最终双向相似度与注意力。
    final_logits, final_debug = model(images, text_ids, text_mask, return_debug=True)  # 对六个 pair 执行最终检索前向。
print("CLIP训练轨迹=", history)  # 展示双向 loss、R@1、温度与梯度变化。
print("归一化embedding norms image/text=", final_debug["image_embeddings"].norm(dim=1).tolist(), final_debug["text_embeddings"].norm(dim=1).tolist())  # 展示 L2 合同。
print("pair-0文本head0 attention=", torch.round(final_debug["text_attention"][0, 0] * 1000) / 1000)  # 展示四 token 自注意力矩阵。
print("最终图文logits矩阵=", torch.round(final_logits * 100) / 100)  # 展示对角匹配与负样本间隔。

CLIP训练轨迹= [{'step': 0, 'loss': 1.8308727741241455, 'i2t_loss': 1.8445616960525513, 't2i_loss': 1.8171838521957397, 'i2t_r1': 0.1666666716337204, 'scale': 14.285714149475098, 'gradient_norm': 0.7018290537525032}, {'step': 90, 'loss': 0.00010946529801003635, 'i2t_loss': 0.00010737844422692433, 't2i_loss': 0.00011155215179314837, 'i2t_r1': 1.0, 'scale': 15.827052116394043, 'gradient_norm': 0.0022740823520257516}, {'step': 180, 'loss': 3.897053829859942e-05, 'i2t_loss': 3.9337974158115685e-05, 't2i_loss': 3.860310243908316e-05, 'i2t_r1': 1.0, 'scale': 15.918561935424805, 'gradient_norm': 0.0007300511628179681}, {'step': 270, 'loss': 2.4666023819008842e-05, 'i2t_loss': 2.4556726202717982e-05, 't2i_loss': 2.4775321435299702e-05, 'i2t_r1': 1.0, 'scale': 15.973353385925293, 'gradient_norm': 0.0004315278447439918}, {'step': 359, 'loss': 1.7881211533676833e-05, 'i2t_loss': 1.7682525140116923e-05, 't2i_loss': 1.807989610824734e-05, 'i2t_r1': 1.0, 'scale': 16.019546508789062, 'gradient_norm': 0.00

## 4. 双向逐查询结果与结果解读

对同一六对数据分别做图找文和文找图，输出 top1、正例分数、最难负例和是否正确，并与统计基线的双向 Recall@1 比较。

In [4]:
final_image_to_text = final_logits.argmax(dim=1)  # 取得每张图最高分 caption。
final_text_to_image = final_logits.argmax(dim=0)  # 取得每条 caption 最高分图像。
clip_i2t_recall = float((final_image_to_text == pair_indices).to(torch.float32).mean().item())  # 计算图找文 Recall@1。
clip_t2i_recall = float((final_text_to_image == pair_indices).to(torch.float32).mean().item())  # 计算文找图 Recall@1。
print("image query      retrieved text    positive_logit  hardest_negative  correct")  # 输出图找文逐查询表头。
for index, name in enumerate(label_names):  # 逐图展示检索结果和难负例。
    hardest_negative = max(final_logits[index, other].item() for other in range(len(images)) if other != index)  # 找到当前图最高负 caption 分数。
    print(f"{name:<15} {label_names[final_image_to_text[index]]:<16} {final_logits[index, index].item():>14.3f} {hardest_negative:>17.3f} {bool(final_image_to_text[index] == index)}")  # 输出图找文证据。
print("text query       retrieved image   positive_logit  hardest_negative  correct")  # 输出文找图逐查询表头。
for index, name in enumerate(label_names):  # 逐 caption 展示反向检索结果。
    hardest_negative = max(final_logits[other, index].item() for other in range(len(images)) if other != index)  # 找到当前文本最高负图分数。
    print(f"{name:<15} {label_names[final_text_to_image[index]]:<16} {final_logits[index, index].item():>14.3f} {hardest_negative:>17.3f} {bool(final_text_to_image[index] == index)}")  # 输出文找图证据。
print(f"结果解读：统计baseline I2T/T2I R@1={baseline_i2t_recall:.3f}/{baseline_t2i_recall:.3f}；手写CLIP={clip_i2t_recall:.3f}/{clip_t2i_recall:.3f}。")  # 解释双编码器在受控图文数据上的收益。

image query      retrieved text    positive_logit  hardest_negative  correct
横向条码            横向条码                     15.537             3.061 True
纵向条码            纵向条码                     14.625             1.880 True
主对角标            主对角标                     15.260             3.692 True
副对角标            副对角标                     14.825             3.635 True
十字定位标           十字定位标                    15.149             3.518 True
边框标签            边框标签                     14.411             3.382 True
text query       retrieved image   positive_logit  hardest_negative  correct
横向条码            横向条码                     15.537             3.518 True
纵向条码            纵向条码                     14.625             3.061 True
主对角标            主对角标                     15.260             3.635 True
副对角标            副对角标                     14.825             3.692 True
十字定位标           十字定位标                    15.149             3.325 True
边框标签            边框标签                     14.411             3.211

## 5. 失败案例与修正：未归一化点积被向量范数劫持

构造一个图像查询、一个方向匹配的正确文本和一个方向错误但范数巨大的文本。原始 dot-product 会选大范数错误候选；L2 normalize 后 cosine 只比较方向，恢复正确匹配。

In [5]:
query_embedding = torch.tensor([1.0, 0.0])  # 构造指向横轴的图像查询。
candidate_embeddings = torch.tensor([[0.9, 0.1], [2.0, 10.0]])  # 构造方向正确小范数与方向错误大范数文本。
raw_dot_scores = query_embedding @ candidate_embeddings.T  # 用未归一化点积计算候选分数。
normalized_query = torch.nn.functional.normalize(query_embedding, dim=0)  # 把查询归一化为单位向量。
normalized_candidates = torch.nn.functional.normalize(candidate_embeddings, dim=1)  # 把两个文本候选逐行归一化。
cosine_scores = normalized_query @ normalized_candidates.T  # 用 cosine 重新比较方向。
raw_choice = int(raw_dot_scores.argmax().item())  # 读取原始点积选择的错误大范数候选。
cosine_choice = int(cosine_scores.argmax().item())  # 读取归一化后选择的正确方向候选。
print(f"错误行为：raw dot scores={raw_dot_scores.tolist()}，choice={raw_choice}，被大范数候选劫持。")  # 展示向量长度污染相似度。
print(f"修正行为：cosine scores={cosine_scores.tolist()}，choice={cosine_choice}，只比较语义方向。")  # 展示 L2 归一化后的正确检索。

错误行为：raw dot scores=[0.8999999761581421, 2.0]，choice=1，被大范数候选劫持。
修正行为：cosine scores=[0.9938837289810181, 0.1961161345243454]，choice=0，只比较语义方向。


## 6. 生产边界

六对规则图文只验证计算图。生产 CLIP 需要来源级 train/test 去重、多 caption/multi-positive loss、海量 in-batch 或跨卡负样本、图文增强、OCR 与多语言评测、版权/隐私/安全过滤、ANN 索引版本、温度校准和按品类监控双向 Recall、hard-negative 与模态漂移。

In [6]:
clip_diagnostics = {"pairs": len(images), "vocabulary": len(vocabulary), "baseline_i2t_r1": baseline_i2t_recall, "baseline_t2i_r1": baseline_t2i_recall, "clip_i2t_r1": clip_i2t_recall, "clip_t2i_r1": clip_t2i_recall, "initial_loss": history[0]["loss"], "final_loss": history[-1]["loss"], "temperature_inverse": final_debug["scale"].item(), "raw_choice": raw_choice, "cosine_choice": cosine_choice}  # 汇总数据、训练、检索与归一化失败指标。
print("生产监控快照：", clip_diagnostics)  # 输出图文检索系统应持续观察的信号。

生产监控快照： {'pairs': 6, 'vocabulary': 13, 'baseline_i2t_r1': 0.1666666716337204, 'baseline_t2i_r1': 0.1666666716337204, 'clip_i2t_r1': 1.0, 'clip_t2i_r1': 1.0, 'initial_loss': 1.8308727741241455, 'final_loss': 1.7881211533676833e-05, 'temperature_inverse': 16.020036697387695, 'raw_choice': 1, 'cosine_choice': 0}


## 7. 最小回归测试

最后一格只保护 pair 规模、真实训练、单位 embedding、双向检索收益和 L2 修正。

In [7]:
assert len(images) >= 5 and images.shape == (6, 1, 16, 16) and text_ids.shape == (6, 4)  # 保证包含足够多可读图文 pair。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证双塔与温度真实 backward 学习。
assert torch.allclose(final_debug["image_embeddings"].norm(dim=1), torch.ones(6), atol=1.0e-5) and torch.allclose(final_debug["text_embeddings"].norm(dim=1), torch.ones(6), atol=1.0e-5)  # 保证两个模态投影后单位归一化。
assert clip_i2t_recall > baseline_i2t_recall and clip_t2i_recall > baseline_t2i_recall  # 保证双向同数据检索都优于统计基线。
assert clip_i2t_recall >= 0.80 and clip_t2i_recall >= 0.80  # 保证绝大多数受控 pair 双向 top1 正确。
assert raw_choice == 1 and cosine_choice == 0  # 保证范数劫持失败可复现并被 cosine 修正。